# Bias-Variance Tradeoff — Exercises

Companion to the note [Bias-Variance Tradeoff](Bias-Variance%20Tradeoff.md).

Practise the decomposition $\mathbb E[(y-\hat f(x))^2] = \text{bias}^2 + \text{variance} + \sigma^2$: derive it, compute it by hand for simple estimators (shrinkage, k-NN, ridge, bagging), estimate it by Monte Carlo simulation, read learning curves, and reproduce double descent.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression, Ridge
rng = np.random.default_rng(0)
f_true = lambda x: np.sin(2 * np.pi * x)
SIGMA = 0.3
x_grid = np.linspace(0.05, 0.95, 50)

def sample(n, r):
    x = r.uniform(0, 1, n)
    return x, f_true(x) + SIGMA * r.normal(size=n)

## Part A · Concepts

### Exercise 1 · Diagnose and fix
*🧠 Concept · ★☆☆*

Use the note's table to classify each case as high bias or high variance and pick a fix.

(a) A linear model for house prices: train RMSE 80k, validation RMSE 82k, target std 90k. (b) A depth-30 decision tree: train accuracy 100 %, validation 71 %.
(c) A 1-NN classifier trained on 50 images. (d) A neural net whose validation error keeps dropping as you add data, with a large train/val gap.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

High bias: both errors high, small gap. High variance: low train error, large gap.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **High bias**: richer model or better features (interactions, non-linear terms). (b) **High variance**: prune / limit depth, or bag the trees into a random forest
([[Ensemble Methods]]). (c) **High variance**: increase $k$, or get more data. (d) **High variance** that more data is fixing: collect more data, plus
regularization / augmentation ([[Overfitting and Regularization]]).

</details>

### Exercise 2 · Which way do the knobs turn?
*🧠 Concept · ★★☆*

For each change say whether bias and variance typically go up (↑), down (↓) or stay (–):
(a) more training data; (b) larger $k$ in k-NN; (c) larger $\lambda$ in ridge; (d) higher polynomial degree; (e) bagging deep trees; (f) boosting stumps for more rounds.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Bias is about the *average* fit of the model family; variance about how much the fit moves between training sets.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Change | Bias | Variance |
|---|---|---|
| (a) more data | – (fixed model family) | ↓ |
| (b) larger $k$ | ↑ (averages over a wider, less local region) | ↓ (averages $k$ noisy targets) |
| (c) larger $\lambda$ | ↑ | ↓ |
| (d) higher degree | ↓ | ↑ |
| (e) bagging | – (≈ same as one tree) | ↓ |
| (f) boosting rounds | ↓ (sequentially fixes residuals) | ↑ slowly |

</details>

### Exercise 3 · Irreducible error
*🧠 Concept · ★★☆*

(a) Why can no model beat $\sigma^2$ on average? (b) Can $\sigma^2$ shrink if you collect **new features**? (c) A model has test MSE below the estimated noise variance.
What does that suggest?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma^2$ is $\mathrm{Var}(y\mid x)$: it depends on what $x$ contains.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Even the true $f(x) = \mathbb E[y\mid x]$ incurs $\mathbb E[(y - f(x))^2] = \sigma^2$; the decomposition's other terms are non-negative.
(b) **Yes**: the noise is irreducible *given these features*. A new informative feature changes the conditioning set and can explain part of what was noise.
(c) Either the noise estimate is wrong, or (often) **leakage**: the model sees information it should not have ([[Cross-Validation and Model Selection]]).

</details>

### Exercise 4 · Double descent
*🧠 Concept · ★★★*

Explain the "modern caveat" in the note. Why does the test error of minimum-norm least squares **peak** when the number of parameters $p$ equals the number of training samples $n$,
and why can it **decrease** again for $p \gg n$? Does this contradict the bias–variance tradeoff?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

At $p=n$ there is exactly one interpolating solution; for $p>n$ there are infinitely many and the solver picks the one with minimum norm.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At $p\approx n$ the design matrix is square and badly conditioned: the unique interpolant needs huge coefficients to fit the noise, so the **variance explodes**.
For $p > n$ many interpolants exist; the minimum-norm one spreads weight over many features, which acts as implicit regularization, so variance falls again while
richer features lower bias. The decomposition still holds; what fails is the assumption that "capacity = number of parameters" (the effective complexity is the norm,
not $p$). You will reproduce this in the last exercise.

</details>

## Part B · By hand

### Exercise 5 · Derive the decomposition
*✍️ By hand · ★★☆*

Let $y = f(x) + \varepsilon$ with $\mathbb E\varepsilon = 0$, $\mathrm{Var}\,\varepsilon = \sigma^2$, and $\varepsilon$ independent of the training set $\mathcal D$ that determines $\hat f$.
Prove the note's formula for $\mathbb E_{\mathcal D,\varepsilon}[(y - \hat f(x))^2]$ at a fixed $x$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Add and subtract $f(x)$ and then $\bar f(x) = \mathbb E_{\mathcal D}[\hat f(x)]$; the cross terms have zero expectation.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Write $y - \hat f = \varepsilon + (f - \bar f) + (\bar f - \hat f)$. Squaring and taking expectations:
- $\mathbb E[\varepsilon^2] = \sigma^2$;
- $(f-\bar f)^2$ is a constant: **bias²**;
- $\mathbb E[(\bar f - \hat f)^2]$ = **variance**;
- cross terms: $\mathbb E[\varepsilon(\cdot)] = 0$ by independence and $\mathbb E\varepsilon = 0$; $2(f-\bar f)\,\mathbb E[\bar f-\hat f] = 0$ since $\mathbb E\hat f = \bar f$.

Hence $\mathbb E[(y-\hat f)^2] = (\bar f - f)^2 + \mathbb E[(\hat f - \bar f)^2] + \sigma^2$. ∎

</details>

### Exercise 6 · Decomposition from four training sets
*✍️ By hand · ★☆☆*

At a point $x_0$ with $f(x_0) = 2$ and $\sigma^2 = 0.25$, a model trained on four different datasets predicts $\hat f(x_0) = (1.5, 2.5, 3.0, 2.0)$.
Treating these four as the whole distribution over datasets, compute bias², variance and the expected squared error at $x_0$.

In [ ]:
bias2 = None
variance = None
expected_err = None

_p = np.array([1.5, 2.5, 3.0, 2.0])
# brute force: average squared error over all datasets and a fine grid of noise quantiles
_eps = np.sqrt(0.25) * np.sqrt(2) * __import__('scipy').special.erfinv(np.linspace(-1, 1, 200001)[1:-1])
check('bias^2', bias2, (_p.mean() - 2) ** 2); check('variance', variance, _p.var())
check('expected error', expected_err, np.mean([np.mean((2 + _eps - p) ** 2) for p in _p]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\bar f(x_0) = 2.25$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\bar f = 9/4 = 2.25$, bias² $= 0.25^2 = 0.0625$, variance $= \frac14(0.5625 + 0.0625 + 0.5625 + 0.0625) = 0.3125$,
expected error $= 0.0625 + 0.3125 + 0.25 = 0.625$. Here variance dominates.

```python
bias2 = 0.0625
variance = 0.3125
expected_err = 0.625
```

</details>

### Exercise 7 · A biased estimator that wins
*✍️ By hand · ★★☆*

Estimate a mean $\mu$ from $n$ i.i.d. samples with variance $\sigma^2$ using the shrunken estimator $\hat\mu_c = c\,\bar x$, $0\le c\le1$.
(a) Give its bias, variance and MSE. (b) Find the $c^*$ minimising the MSE. (c) For $\mu = 1$, $\sigma^2 = 4$, $n = 4$, compute $c^*$ and compare MSE$(c^*)$ with MSE$(1)$.

In [ ]:
c_star = None
mse_cstar = None
mse_unbiased = None

_r = np.random.default_rng(6); _xb = _r.normal(1, 2, size=(400000, 4)).mean(1)
check('c*', c_star, 1 / (1 + 4 / 4))
if c_star is not None:
    check('MSE(c*) (simulated)', mse_cstar, np.mean((c_star * _xb - 1) ** 2), tol=1e-2)
check('MSE(1) (simulated)', mse_unbiased, np.mean((_xb - 1) ** 2), tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

Bias $(c-1)\mu$, variance $c^2\sigma^2/n$. Differentiate the sum w.r.t. $c$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) MSE$(c) = (c-1)^2\mu^2 + c^2\sigma^2/n$. (b) $2(c-1)\mu^2 + 2c\sigma^2/n = 0 \Rightarrow c^* = \dfrac{\mu^2}{\mu^2 + \sigma^2/n}$.
(c) $\sigma^2/n = 1$, so $c^* = 0.5$: MSE $= 0.25 + 0.25 = 0.5$, versus $1$ for the unbiased $\bar x$. Accepting bias to cut variance halves the error;
this is exactly what ridge does. (In practice $c^*$ depends on the unknown $\mu$, which is why $\lambda$ is tuned by CV.)

```python
c_star = 0.5
mse_cstar = 0.5
mse_unbiased = 1.0
```

</details>

### Exercise 8 · Bias and variance of k-NN
*✍️ By hand · ★★☆*

Training inputs lie on a grid with spacing $h = 0.1$, the target is $f(x) = x^2$ with noise variance $\sigma^2 = 1$, and k-NN with $k = 2m+1$ averages the targets at
$x_0 + jh$, $j = -m..m$. (a) Show bias $= h^2\,m(m+1)/3$ and variance $= \sigma^2/k$. (b) Compute the expected squared error at $x_0$ for $k=5$.
(c) For which $k$ is the sum bias² + variance smallest (try $k = 1, 3, \dots, 61$)?

In [ ]:
err_k5 = None
best_k = None

def _err(k, h=0.1, s2=1.0, x0=0.3):
    m = (k - 1) // 2; js = np.arange(-m, m + 1)
    bias = np.mean((x0 + js * h) ** 2) - x0 ** 2
    return bias ** 2 + s2 / k + s2
check('error at k=5', err_k5, _err(5))
_ks = np.arange(1, 62, 2)
check('best k', best_k, _ks[int(np.argmin([_err(k) for k in _ks]))])

<details>
<summary><b>💡 Hint 1</b></summary>

$\frac1k\sum_j (x_0 + jh)^2 = x_0^2 + \frac{2x_0h}{k}\sum_j j + \frac{h^2}{k}\sum_j j^2$ and $\sum_{j=-m}^m j^2 = \frac{m(m+1)(2m+1)}{3}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Minimise $g(m) = h^4 m^2(m+1)^2/9 + \sigma^2/(2m+1)$ over integers numerically.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The linear term cancels by symmetry, leaving bias $= \frac{h^2}{k}\cdot\frac{m(m+1)(2m+1)}{3} = h^2\frac{m(m+1)}{3}$. Averaging $k$ independent noises gives $\sigma^2/k$.
(b) $m=2$: bias $= 0.01\cdot2 = 0.02$, bias² $= 0.0004$, variance $= 0.2$, total $= 0.0004 + 0.2 + 1 = 1.2004$.
(c) Bias grows like $m^4$, variance shrinks like $1/m$; scanning gives $k^* = 13$ ($m=6$): bias² $= 0.14^2 \approx 0.0196$, variance $= 1/13 \approx 0.077$. Larger $k$ = smoother = more bias.

```python
err_k5 = 0.02 ** 2 + 1 / 5 + 1
_cands = np.arange(1, 62, 2)
_tot = [(0.01 * ((k - 1) // 2) * ((k - 1) // 2 + 1) / 3) ** 2 + 1 / k for k in _cands]
best_k = int(_cands[int(np.argmin(_tot))])
```

</details>

### Exercise 9 · Bagging and correlation
*✍️ By hand · ★★☆*

Average $B$ models, each with prediction variance $\sigma^2$ and pairwise correlation $\rho$. Show that the average has variance $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$.
Evaluate for $\sigma^2 = 1$, $\rho = 0.3$, $B = 10$ and $B\to\infty$. What does a random forest do to $\rho$?

In [ ]:
var_B10 = None
var_Binf = None

_C = 0.3 * np.ones((10, 10)) + 0.7 * np.eye(10)         # covariance of the 10 models
check('B = 10', var_B10, np.ones(10) @ _C @ np.ones(10) / 100)
check('B -> inf', var_Binf, 0.3)

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{Var}\big(\frac1B\sum_b Z_b\big) = \frac{1}{B^2}\sum_{b,b'}\mathrm{Cov}(Z_b, Z_{b'})$, with $B$ diagonal and $B(B-1)$ off-diagonal terms.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{1}{B^2}\big(B\sigma^2 + B(B-1)\rho\sigma^2\big) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$. $B=10$: $0.3 + 0.07 = 0.37$; $B\to\infty$: $0.3$.
Averaging only removes the uncorrelated part: the floor $\rho\sigma^2$ remains. Random forests subsample features at each split to **decorrelate** the trees, lowering that floor.

```python
var_B10 = 0.37
var_Binf = 0.3
```

</details>

### Exercise 10 · Bias and variance of 1-D ridge
*✍️ By hand · ★★★*

Model $y_i = w x_i + \varepsilon_i$ with fixed inputs, $S = \sum_i x_i^2$, and the ridge estimator $\hat w_\lambda = \frac{\sum_i x_iy_i}{S+\lambda}$.
(a) Derive $\mathbb E[\hat w_\lambda]$ and $\mathrm{Var}[\hat w_\lambda]$. (b) For $S = 10$, $w = 2$, $\sigma^2 = 1$, compute bias², variance and MSE of $\hat w$ for $\lambda = 1$.
(c) Show that the MSE-optimal penalty is $\lambda^* = \sigma^2/w^2$ and evaluate it.

In [ ]:
bias2_l1 = None
var_l1 = None
lam_star = None

_S, _w, _s2 = 10.0, 2.0, 1.0
_mse = lambda l: (_w * _S / (_S + l) - _w) ** 2 + _s2 * _S / (_S + l) ** 2
check('bias^2 at lambda=1', bias2_l1, (_w * _S / 11 - _w) ** 2)
check('variance at lambda=1', var_l1, _s2 * _S / 121)
_ls = np.linspace(0, 2, 200001)
check('lambda*', lam_star, _ls[np.argmin(_mse(_ls))], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\sum x_iy_i = wS + \sum x_i\varepsilon_i$, and $\mathrm{Var}(\sum x_i\varepsilon_i) = \sigma^2 S$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathbb E\hat w = \frac{wS}{S+\lambda}$, bias $= -\frac{\lambda w}{S+\lambda}$, $\mathrm{Var} = \frac{\sigma^2 S}{(S+\lambda)^2}$.
(b) $\lambda=1$: bias² $= (20/11 - 2)^2 = (2/11)^2 \approx 0.0331$, variance $= 10/121 \approx 0.0826$, MSE $\approx 0.1157$ — slightly *worse* than OLS ($0.1$).
(c) $\frac{d}{d\lambda}\Big[\frac{\lambda^2w^2 + \sigma^2S}{(S+\lambda)^2}\Big] = 0 \Rightarrow \lambda w^2 S = \sigma^2 S \Rightarrow \lambda^* = \sigma^2/w^2 = 0.25$, giving MSE $\approx 0.0976 < 0.1$.
Some shrinkage always helps, too much hurts: the optimum depends on the signal-to-noise ratio.

```python
bias2_l1 = (2 / 11) ** 2
var_l1 = 10 / 121
lam_star = 0.25
```

</details>

## Part C · Simulation

### Exercise 11 · Monte Carlo bias–variance decomposition
*💻 Code · ★★☆*

Using `sample(n, r)` from the setup ($f = \sin 2\pi x$, $\sigma=0.3$), draw 300 training sets of size $n = 30$ and fit polynomials of degree $d\in\{1, 3, 9\}$ by least squares.
At the points `x_grid` estimate bias² and variance (averaged over the grid) and store them in dicts `bias2_d`, `var_d`. Also estimate the expected test MSE `mse_d`
directly with fresh noisy targets at `x_grid`, and check that bias² + variance + $\sigma^2$ matches it.

In [ ]:
bias2_d = None
var_d = None
mse_d = None

if bias2_d is not None:
    check('decomposition adds up (within 5%)', all(abs(bias2_d[d] + var_d[d] + SIGMA**2 - mse_d[d]) < 0.05 * mse_d[d] for d in (1, 3, 9)), True)
    check('degree 1 has the largest bias', max(bias2_d, key=bias2_d.get), 1)
    check('degree 9 has the largest variance', max(var_d, key=var_d.get), 9)
else:
    check('decomposition', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Collect predictions in an array `P` of shape (300, 50) per degree; then `bias2 = np.mean((P.mean(0) - f_true(x_grid)) ** 2)`, `var = np.mean(P.var(0))`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For the direct MSE: `y_new = f_true(x_grid) + SIGMA * r.normal(size=(300, 50))`, `np.mean((P - y_new) ** 2)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Degree 1 is dominated by bias² (≈ 0.15), degree 9 by variance (which explodes near the interval ends), degree 3 balances both and has the lowest total. The Monte Carlo MSE matches
bias² + variance + $\sigma^2$ up to simulation error, which is the note's formula averaged over $x$.

```python
r = np.random.default_rng(10)
bias2_d, var_d, mse_d = {}, {}, {}
for d in (1, 3, 9):
    P = np.empty((300, len(x_grid)))
    for t in range(300):
        x, y = sample(30, r)
        w = np.linalg.lstsq(np.vander(x, d + 1), y, rcond=None)[0]
        P[t] = np.vander(x_grid, d + 1) @ w
    y_new = f_true(x_grid) + SIGMA * r.normal(size=P.shape)
    bias2_d[d] = np.mean((P.mean(0) - f_true(x_grid)) ** 2)
    var_d[d] = np.mean(P.var(0))
    mse_d[d] = np.mean((P - y_new) ** 2)
    print(d, round(bias2_d[d], 4), round(var_d[d], 4), round(bias2_d[d] + var_d[d] + SIGMA**2, 4), round(mse_d[d], 4))
```

</details>

### Exercise 12 · k in k-NN controls the tradeoff
*💻 Code · ★★☆*

Repeat the simulation for `KNeighborsRegressor(k)` with $n = 100$, 200 training sets and $k \in \{1, 5, 20, 50\}$. Store lists `knn_bias2` and `knn_var` (one value per $k$).

In [ ]:
ks = [1, 5, 20, 50]
knn_bias2 = None
knn_var = None

if knn_bias2 is not None:
    check('bias^2 grows for large k', knn_bias2[3] > knn_bias2[2] > max(knn_bias2[:2]), True)
    check('variance falls from k=1 to k=20', bool(np.all(np.diff(knn_var[:3]) < 0)), True)
else:
    check('k-NN tradeoff', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Same loop as before, with `KNeighborsRegressor(k).fit(x[:, None], y).predict(x_grid[:, None])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Variance drops roughly like $\sigma^2/k$ at first ($0.09 \to 0.019 \to 0.008$) while bias² grows as the neighbourhood widens over the curved sine ($\approx0$, $0.006$, $0.066$).
$k=1$ is pure variance; $k=50$ (half the data) is mostly bias. At $k=50$ the variance even ticks up slightly: *which* points fall into such a wide neighbourhood
changes from dataset to dataset, and with a curved $f$ that adds variance. (Note also that a Monte Carlo bias² estimate carries an upward error of about variance/#datasets,
so tiny bias values cannot be ordered reliably.)

```python
r = np.random.default_rng(11)
ks = [1, 5, 20, 50]
knn_bias2, knn_var = [], []
for k in ks:
    P = np.empty((200, len(x_grid)))
    for t in range(200):
        x, y = sample(100, r)
        P[t] = KNeighborsRegressor(k).fit(x[:, None], y).predict(x_grid[:, None])
    knn_bias2.append(np.mean((P.mean(0) - f_true(x_grid)) ** 2)); knn_var.append(np.mean(P.var(0)))
print(np.round(knn_bias2, 4), np.round(knn_var, 4))
```

</details>

### Exercise 13 · Ridge λ controls the tradeoff
*💻 Code · ★★☆*

With degree-9 polynomial features (columns $x, \dots, x^9$ standardised using the fixed grid's statistics) and $n = 20$, estimate bias² and variance of ridge for
$\lambda\in\{10^{-3}, 10^{-1}, 1, 10\}$ over 300 training sets. Store `ridge_bias2`, `ridge_var` and the $\lambda$ with the smallest bias² + variance in `ridge_best`.

In [ ]:
lams_bv = [1e-3, 1e-1, 1.0, 10.0]
_G = np.vander(x_grid, 10, increasing=True)[:, 1:]
mu_G, sd_G = _G.mean(0), _G.std(0)
def phi(x):
    return (np.vander(x, 10, increasing=True)[:, 1:] - mu_G) / sd_G
ridge_bias2 = None
ridge_var = None
ridge_best = None

if ridge_bias2 is not None:
    check('bias^2 increases with lambda', bool(np.all(np.diff(ridge_bias2) > 0)), True)
    check('variance decreases with lambda', bool(np.all(np.diff(ridge_var) < 0)), True)
    check('best lambda is intermediate', ridge_best in (1e-1, 1.0), True)
else:
    check('ridge tradeoff', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Fit `Ridge(alpha=lam).fit(phi(x), y)` (it fits an unpenalised intercept) and predict at `phi(x_grid)`. Use the same training sets for every $\lambda$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Small $\lambda$: almost unbiased, high variance (wiggly degree-9 fits). Large $\lambda$: nearly constant predictions, large bias. An intermediate $\lambda$
minimises the sum; in practice it is chosen by [[Cross-Validation and Model Selection]].

```python
lams_bv = [1e-3, 1e-1, 1.0, 10.0]
_G = np.vander(x_grid, 10, increasing=True)[:, 1:]
mu_G, sd_G = _G.mean(0), _G.std(0)
def phi(x):
    return (np.vander(x, 10, increasing=True)[:, 1:] - mu_G) / sd_G
r = np.random.default_rng(12)
data = [sample(20, r) for _ in range(300)]
ridge_bias2, ridge_var = [], []
for lam in lams_bv:
    P = np.array([Ridge(alpha=lam).fit(phi(x), y).predict(phi(x_grid)) for x, y in data])
    ridge_bias2.append(np.mean((P.mean(0) - f_true(x_grid)) ** 2)); ridge_var.append(np.mean(P.var(0)))
ridge_best = lams_bv[int(np.argmin(np.add(ridge_bias2, ridge_var)))]
print(np.round(ridge_bias2, 4), np.round(ridge_var, 4), ridge_best)
```

</details>

### Exercise 14 · Bagging reduces variance
*💻 Code · ★★☆*

Compare a fully grown `DecisionTreeRegressor` with `BaggingRegressor(DecisionTreeRegressor(), n_estimators=30)` over 60 training sets of size $n=60$.
Store bias² and variance of each as `tree_bv = (bias2, var)` and `bag_bv = (bias2, var)`.

In [ ]:
tree_bv = None
bag_bv = None

if tree_bv is not None:
    check('bagging cuts variance by > 40%', bag_bv[1] < 0.6 * tree_bv[1], True)
    check('bias stays small', bag_bv[0] < 0.02 and tree_bv[0] < 0.02, True)
else:
    check('bagging', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Give each model a fixed `random_state` inside the loop for reproducibility; trees are deterministic given the data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A deep tree has low bias but its predictions jump with every training set. Averaging bootstrap trees keeps the bias roughly unchanged and removes a large part of the variance,
down to the correlated floor $\rho\sigma^2$ from the bagging exercise.

```python
r = np.random.default_rng(13)
Pt, Pb = np.empty((60, len(x_grid))), np.empty((60, len(x_grid)))
for t in range(60):
    x, y = sample(60, r)
    Pt[t] = DecisionTreeRegressor(random_state=0).fit(x[:, None], y).predict(x_grid[:, None])
    Pb[t] = BaggingRegressor(DecisionTreeRegressor(), n_estimators=30, random_state=t).fit(x[:, None], y).predict(x_grid[:, None])
bv = lambda P: (np.mean((P.mean(0) - f_true(x_grid)) ** 2), np.mean(P.var(0)))
tree_bv, bag_bv = bv(Pt), bv(Pb)
print('tree', np.round(tree_bv, 4), '| bagged', np.round(bag_bv, 4))
```

</details>

### Exercise 15 · Learning curves
*💻 Code · ★★☆*

For training sizes $n\in\{10, 20, 50, 100, 200, 500\}$ compute the training MSE and the validation MSE (on 2000 fresh points) of (a) linear regression and (b) 1-NN regression.
Store dicts `curve_lin`, `curve_1nn` mapping $n$ to `(train_mse, val_mse)`. Which model is high-bias and which high-variance? Would more data help each?

In [ ]:
r_lc = np.random.default_rng(14)
x_val, y_val = sample(2000, r_lc)
sizes = [10, 20, 50, 100, 200, 500]
curve_lin = None
curve_1nn = None

if curve_lin is not None:
    check('linear: both errors high and close at n=500', curve_lin[500][0] > 0.2 and abs(curve_lin[500][1] - curve_lin[500][0]) < 0.05, True)
    check('1-NN: zero training error, val error falls with n', curve_1nn[500][0] < 1e-12 and curve_1nn[500][1] < curve_1nn[10][1], True)
else:
    check('learning curves', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Draw one training set per size with `sample(n, r_lc)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Linear regression: train and validation curves converge quickly to ≈ 0.29 ≫ $\sigma^2 = 0.09$: **high bias**, more data does not help; a richer model does.
1-NN: training error is 0, validation error starts high and keeps falling toward $2\sigma^2$ (1-NN's asymptotic error): **high variance**; more data helps.

```python
r_lc = np.random.default_rng(14)
x_val, y_val = sample(2000, r_lc)
sizes = [10, 20, 50, 100, 200, 500]
curve_lin, curve_1nn = {}, {}
for n in sizes:
    x, y = sample(n, r_lc)
    for model, store in ((LinearRegression(), curve_lin), (KNeighborsRegressor(1), curve_1nn)):
        model.fit(x[:, None], y)
        store[n] = (np.mean((model.predict(x[:, None]) - y) ** 2), np.mean((model.predict(x_val[:, None]) - y_val) ** 2))
for n in sizes:
    print(n, np.round(curve_lin[n], 3), np.round(curve_1nn[n], 3))
```

</details>

### Exercise 16 · Reproduce double descent
*💻 Code · ★★★*

Linear target in $D = 400$ dimensions: $\beta\sim\mathcal N(0, I/D)$, $x\sim\mathcal N(0, I)$, $y = x^\top\beta + 0.3\,\varepsilon$, with only $n = 40$ training samples.
The model uses only the **first $p$** features and is fitted by minimum-norm least squares (`np.linalg.lstsq`). For
$p\in\{5, 10, 20, 30, 36, 40, 44, 50, 80, 160, 400\}$ compute the noiseless test MSE (500 test points), repeat over 20 seeds (`np.random.default_rng(seed)` for seed 0..19),
and store the **median** over seeds in `dd_err` (one value per $p$).

In [ ]:
ps = [5, 10, 20, 30, 36, 40, 44, 50, 80, 160, 400]
dd_err = None

if dd_err is not None:
    check('peak at the interpolation threshold p = n = 40', ps[int(np.argmax(dd_err))], 40)
    check('p = 400 beats every under-parameterised model', dd_err[-1] < min(dd_err[:4]), True)
else:
    check('double descent', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Per seed: draw `beta`, `X` (40×400), `y`, and `X_test` (500×400) once; then loop over $p$ using `X[:, :p]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For $p > n$ `lstsq` returns the minimum-norm interpolant automatically.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Error first rises (classic U: variance grows as $p\to n$), **peaks at $p = n = 40$** where the square design matrix is ill-conditioned and the interpolant fits the noise with huge
weights, then falls again: with $p=400$ the minimum-norm solution has lower error than any $p<n$ model. The usual U-curve is only the left half of the picture
for over-parameterised models.

```python
ps = [5, 10, 20, 30, 36, 40, 44, 50, 80, 160, 400]
E = []
for seed in range(20):
    r = np.random.default_rng(seed)
    beta = r.normal(size=400) / np.sqrt(400)
    X = r.normal(size=(40, 400)); y = X @ beta + 0.3 * r.normal(size=40)
    Xt = r.normal(size=(500, 400)); yt = Xt @ beta
    E.append([np.mean((Xt[:, :p] @ np.linalg.lstsq(X[:, :p], y, rcond=None)[0] - yt) ** 2) for p in ps])
dd_err = np.median(E, axis=0)
print(dict(zip(ps, dd_err.round(3))))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Bias-Variance Tradeoff](Bias-Variance%20Tradeoff.md).*